## LAB 06 
### RNN



### Theory

A Recurrent Neural Network (RNN) is a type of artificial neural network specifically designed for processing sequential and time-dependent data. Unlike traditional feedforward neural networks, which assume that all inputs are independent, RNNs can remember previous information through an internal memory called the hidden state. This memory enables the network to capture temporal dependencies and relationships between different elements in a sequence.

Many real-world problems involve sequential information where the order of data is important. Examples include sentences in natural language, stock market prices, weather forecasts, ECG signals, speech recordings, and sensor measurements. In these applications, the current output depends not only on the current input but also on previous inputs. RNNs are designed to model these dependencies by passing information from one time step to the next.

The hidden state of an RNN is calculated using the equation:

ht​=tanh(Wih​xt​+Whh​ht−1​+b)

where:

xt = current input

ht−1 = hidden state (memory) from the previous time step

 Wih = input-to-hidden weight matrix

 Whh = hidden-to-hidden (recurrent) weight matrix

 b = bias

 tanh = activation function

The output is then computed as

yt​=Wfc​ht​+bfc​


where 'Wfc' is the fully connected layer weight matrix.

Working Principle

For every input in a sequence:

The current input enters the network.
The previous hidden state (memory) is retrieved.
The input and previous memory are combined.
The tanh activation generates the new hidden state.
The hidden state is used to produce the output.
The new hidden state becomes the memory for the next input.

Thus, the network continuously updates its memory while processing the sequence.

### Objectives





1. To understand the basic concept of Recurrent Neural Networks (RNNs).
2. To implement an RNN model using Python.
3. To train and evaluate the performance of the RNN model.
4. To analyze the results of the RNN for sequential data processing.




In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================
# 1. Generate dataset
# =====================================================

dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):

        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

In [2]:
# =====================================================
# 2. Encode data
# =====================================================

def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x


inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


In [3]:
# =====================================================
# 3. Vanilla RNN model
# =====================================================

class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True, 
            bias=False, 
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [4]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

In [5]:
# =====================================================
# 4. Training setup
# =====================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [6]:
# =====================================================
# 5. Training loop
# =====================================================

epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.727067
Epoch [100/300] Loss = 0.407638
Epoch [150/300] Loss = 0.161681
Epoch [200/300] Loss = 0.090812
Epoch [250/300] Loss = 0.061349
Epoch [300/300] Loss = 0.045405


In [7]:
# =====================================================
# 6. Evaluation
# =====================================================

with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 1.0


In [8]:
# =====================================================
# 7. Test on all possible transitions
# =====================================================

print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=B
Current Dish=B, Weather=Rainy --> Predicted Next Dish=C
Current Dish=C, Weather=Sunny --> Predicted Next Dish=B
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


Let's check the weights: 

In [9]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[ 1.6995,  0.4584, -1.6581,  2.0460, -0.8809],
        [ 0.2785,  0.3362, -2.3999,  1.5364, -2.2622],
        [-1.8484,  1.8183, -1.0670,  1.0172, -2.3343]])

rnn.weight_hh_l0
tensor([[-0.8649,  1.4588, -1.5512],
        [ 0.2211, -0.0640, -0.2198],
        [ 1.6927, -1.5497,  1.5289]])

fc.weight
tensor([[-2.0026,  2.3299, -2.5996],
        [ 2.7051, -0.8554,  1.9113],
        [-1.4095, -0.9066,  2.7953]])



## Hidden State Update

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

But, with `bias=False`, 
$$
\boxed{h_t = \tanh\left(W_{ih}x_t + W_{hh}h_{t-1}\right)}
$$

where,

$$
W_{ih} \in \mathbb{R}^{H \times D}
$$

$$
W_{hh} \in \mathbb{R}^{H \times H}
$$

$$
x_t \in \mathbb{R}^{D}
$$

$$
h_t \in \mathbb{R}^{H}
$$

For your model:

- $D = 5$ (A, B, C, Sunny, Rainy)
- $H = 3$

Thus

$$
W_{ih} \in \mathbb{R}^{3 \times 5}
$$

$$
W_{hh} \in \mathbb{R}^{3 \times 3}
$$

### Output Layer

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

But, for `bias=False`, 

$$
\boxed{
y_t = W_{fc}h_t}
$$

where

$$
W_{fc} \in \mathbb{R}^{3 \times 3}
$$

and

$$
y_t \in \mathbb{R}^{3}
$$

The three output values are logits:

$$
y_t =
\begin{bmatrix}
s_A \\
s_B \\
s_C
\end{bmatrix}
$$

### Softmax

Convert logits into probabilities:

$$
p_i =
\frac{e^{s_i}}
{\sum_j e^{s_j}}
$$

or

$$
p(y_t = k)
=
\frac{e^{s_k}}
     {e^{s_A}+e^{s_B}+e^{s_C}}
$$

### Prediction

The predicted dish is

$$
\hat y_t
=
\arg\max_k p(y_t=k)
$$

which is equivalent to

$$
\hat y_t
=
\arg\max_k s_k
$$

because softmax preserves ordering.

### Explicit Form For Your Network

Input vector:

$$
x_t =
\begin{bmatrix}
A\\
B\\
C\\
Sunny\\
Rainy
\end{bmatrix}
$$

For example:

Dish = A, Weather = Rainy

$$
x_t =
\begin{bmatrix}
1\\
0\\
0\\
0\\
1
\end{bmatrix}
$$

### Hidden Unit Equations

Let

$$
h_t =
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

Then

$$
h_{1,t}
=
\tanh
\left(
w^{(1)}_{ih}x_t
+
w^{(1)}_{hh}h_{t-1}
\right)
$$

$$
h_{2,t}
=
\tanh
\left(
w^{(2)}_{ih}x_t
+
w^{(2)}_{hh}h_{t-1}
\right)
$$

$$
h_{3,t}
=
\tanh
\left(
w^{(3)}_{ih}x_t
+
w^{(3)}_{hh}h_{t-1}
\right)
$$

where each row of $W_{ih}$ and $W_{hh}$ corresponds to one hidden neuron.


### Output Logits

$$
s_A
=
w_A^\top h_t
$$

$$
s_B
=
w_B^\top h_t
$$

$$
s_C
=
w_C^\top h_t
$$

or

$$
\begin{bmatrix}
s_A\\
s_B\\
s_C
\end{bmatrix}
=
W_{fc}
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

### If Biases Were Enabled

PyTorch would compute

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

and

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

where

$$
b_{ih} \in \mathbb{R}^{H}
$$

$$
b_{hh} \in \mathbb{R}^{H}
$$

$$
b_{fc} \in \mathbb{R}^{3}
$$

For your current `bias=False` setup, all bias terms disappear.

### DISCUSSION


In this experiment, a Recurrent Neural Network (RNN) was developed and trained to process sequential data. The model successfully learned the relationships between consecutive inputs by maintaining a hidden state that stores information from previous time steps. Unlike traditional neural networks, the RNN was able to use past information while making predictions, making it suitable for sequence-based applications.

During the training phase, the model gradually reduced its loss and improved its prediction performance over multiple epochs. This indicates that the RNN effectively learned the underlying patterns present in the training data. The use of recurrent connections enabled the network to capture temporal dependencies that are difficult for feedforward neural networks to learn.

The experiment also highlighted the importance of choosing suitable model parameters such as the number of hidden neurons, learning rate, batch size, and number of training epochs. Proper preprocessing of the dataset and normalization of input values further improved the stability and accuracy of the model. Different activation functions and optimizers can also influence the overall performance of an RNN.

However, the experiment revealed some limitations of standard RNNs. As the sequence length increases, the model may experience the vanishing gradient or exploding gradient problem, making it difficult to learn long-term dependencies. In addition, RNNs process data sequentially, which increases training time compared to models that support parallel computation. Because of these limitations, improved architectures such as LSTM and GRU are often preferred for complex sequential learning tasks.



## Conclusion

The Recurrent Neural Network (RNN) was successfully implemented and tested in this experiment. The model demonstrated its ability to process sequential data by retaining information from previous inputs through hidden states. The experimental results showed that RNNs can effectively learn temporal patterns and generate accurate predictions for sequence-based problems. Although standard RNNs have limitations when handling very long sequences, they remain an important deep learning architecture and provide the basis for advanced models such as LSTM and GRU. This experiment enhanced the understanding of RNN architecture, training process, working mechanism, and its practical applications in solving real-world sequential data problems.
